In [2]:
from confluent_kafka import Consumer, KafkaException, KafkaError
import threading

# Configurer le consommateur Kafka
consumer = Consumer({
    'bootstrap.servers': 'localhost:9092',  # ou 'localhost:9092' si tu n'es pas en Docker
    'group.id': 'my-consumer-group',
    'auto.offset.reset': 'earliest',  # Ou 'latest' si tu veux commencer à consommer les derniers messages
})

# Le nom du topic auquel tu veux souscrire
topic_name = "data_topic"  # Remplace par le nom de ton topic Kafka

# S'abonner au topic
consumer.subscribe([topic_name])

def process_kafka_messages():
    try:
        while True:
            # Consommer un message
            msg = consumer.poll(1.0)  # Attendre 1 seconde pour un message

            if msg is None:
                # Aucun message disponible
                continue
            if msg.error():
                # Erreur lors de la consommation du message
                if msg.error().code() == KafkaError._PARTITION_EOF:
                    print(f"Fin de partition atteint {msg.partition} {msg.offset}")
                else:
                    raise KafkaException(msg.error())
            else:
                # Message consommé avec succès
                print(f"Reçu message : {msg.value().decode('utf-8')}")

    except Exception as e:
        print(f"Erreur dans le consommateur Kafka: {str(e)}")
    finally:
        consumer.close()  # Fermer le consommateur proprement

# Démarrer le consommateur Kafka dans un thread séparé
def start_kafka_consumer():
    thread = threading.Thread(target=process_kafka_messages, daemon=True)
    thread.start()



In [4]:
from confluent_kafka import Producer


def delivery_report(err, msg):
    """Callback pour savoir si le message a été envoyé avec succès"""
    if err is not None:
        print(f"Erreur d'envoi Kafka: {err}")
    else:
        return (f"Message envoyé à {msg.topic()} [{msg.partition()}] avec offset {msg.offset()}")


KAFKA_BROKER = "localhost:9092"  # Utilise "kafka" qui est le nom du service dans Docker Compose

producer = Producer({
    'bootstrap.servers': KAFKA_BROKER,  # Connexion au broker Kafka
    'client.id': 'my-producer',
})



